In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModel
import matplotlib.pyplot as plt

# ======================
# CONFIG
# ======================
DATA_PATH = "../clean_data/test_cleaned.csv"   # or test.csv
IMAGES_DIR = "../images/test"
MODEL_NAME = "google/siglip-base-patch16-224"   # Vision encoder
IMAGE_EMB_PATH = "test_image_embeddings_siglip.npy"
DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} samples")

# ======================
# LOAD MODEL (VISION ENCODER)
# ======================
print("\n🔹 Loading SigLIP Vision model...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True).vision_model.to(DEVICE)
model.eval()

# ======================
# HELPER: Load image
# ======================
def load_image(sample_id):
    for ext in [".jpg", ".png", ".jpeg", ".webp"]:
        path = os.path.join(IMAGES_DIR, f"{sample_id}{ext}")
        if os.path.exists(path):
            try:
                return Image.open(path).convert("RGB")
            except:
                pass
    return Image.new("RGB", (224, 224), (255, 255, 255))

# ======================
# GENERATE EMBEDDINGS
# ======================
embeddings = []
print(f"\n🖼 Generating embeddings for {len(df)} images in '{IMAGES_DIR}' ...")

for sample_id in tqdm(df["sample_id"].values):
    img = load_image(sample_id)
    
    inputs = processor(images=img, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)
        # Take mean pooling across patches
        emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy().squeeze()
    embeddings.append(emb)

embeddings = np.vstack(embeddings)

# ======================
# SAVE
# ======================
np.save(IMAGE_EMB_PATH, embeddings)
print(f"\n✅ Saved image embeddings → {IMAGE_EMB_PATH}")
print(f"Shape: {embeddings.shape}")

Loaded 75000 samples

🔹 Loading SigLIP Vision model...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]


🖼 Generating embeddings for 75000 images in '../images/test' ...


100%|██████████| 75000/75000 [42:15<00:00, 29.58it/s]



✅ Saved image embeddings → test_image_embeddings_siglip.npy
Shape: (75000, 768)


In [2]:
print(IMAGE_EMB_PATH)

test_image_embeddings_siglip.npy
